# Eliminar NANS y Dividir entre Global y Transect

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# Limpieza de datos
#   Elimina duplicados de datetime
#   Elimina todas las filas con cualquier NaN
#   Mantiene la misma entrada y salida original
#   Muestra:
#       número de NaNs
#       dónde estaban
#       número de duplicados
#       dónde estaban
#       tamaños antes y después
# =============================================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
INPUT_DIR = BASE_DIR

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Columnas que suelen ser numéricas en estos CSV.
# Se convierten a numérico para que valores raros se detecten como NaN.
NUM_COLS = [
    "NO", "NO2", "NOx", "O3_for_impute", "O3",
    "Veloc.", "Direc.", "Temp.", "R.Sol.",
    "Dist.", "Angulo"
]

# Cuántas ubicaciones mostrar por columna / por tipo para no saturar la consola
MAX_SHOW_PER_COLUMN = 10
MAX_SHOW_DUPLICATE_TIMESTAMPS = 20

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime sin modificar todavía duplicados ni NaNs."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)

    # Asegurar DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")

    return df


def detect_transect_label(df):
    """Devuelve el primer valor no nulo de la columna Transecto, o None si no existe."""
    if "Transecto" not in df.columns:
        return None
    tran_names = df["Transecto"].dropna().unique()
    if len(tran_names) == 0:
        return None
    return tran_names[0]


def convert_numeric_columns(df):
    """Convierte a numérico las columnas de NUM_COLS que existan."""
    df = df.copy()
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def report_nans(df, label, max_show_per_column=MAX_SHOW_PER_COLUMN):
    """
    Muestra:
      total de NaNs
      NaNs por columna
      primeras ubicaciones por columna
    """
    nan_mask = df.isna()
    total_nans = int(nan_mask.sum().sum())
    nans_by_column = nan_mask.sum()
    nans_by_column = nans_by_column[nans_by_column > 0].sort_values(ascending=False)

    print(f"\n[{label}] NaNs detectados: {total_nans}")

    if total_nans == 0:
        print(f"[{label}] No hay NaNs en el DataFrame.")
        return total_nans

    print(f"[{label}] NaNs por columna:")
    for col, n in nans_by_column.items():
        positions = df.index[nan_mask[col]].tolist()
        preview = positions[:max_show_per_column]
        preview_str = ", ".join(str(x) for x in preview)
        more = "" if len(positions) <= max_show_per_column else f" ... (+{len(positions) - max_show_per_column} más)"
        print(f"  - {col}: {int(n)} -> [{preview_str}]{more}")

    return total_nans


def report_duplicates(df, label, max_show=MAX_SHOW_DUPLICATE_TIMESTAMPS):
    """
    Muestra:
      número de timestamps duplicados
      cuántas filas están implicadas
      dónde están
    """
    dup_mask = df.index.duplicated(keep=False)
    dup_rows = int(dup_mask.sum())

    if dup_rows == 0:
        print(f"\n[{label}] No hay duplicados de datetime.")
        return {
            "duplicate_rows": 0,
            "duplicate_timestamps": 0,
            "duplicate_locations": []
        }

    duplicated_index_values = df.index[dup_mask]
    dup_counts = pd.Series(duplicated_index_values).value_counts().sort_index()
    duplicate_timestamps = int(len(dup_counts))

    print(f"\n[{label}] Duplicados de datetime detectados:")
    print(f"  - Filas implicadas en duplicados: {dup_rows}")
    print(f"  - Timestamps duplicados únicos: {duplicate_timestamps}")

    print(f"[{label}] Primeros timestamps duplicados y sus posiciones:")
    preview_timestamps = list(dup_counts.index[:max_show])

    duplicate_locations = []
    for ts in preview_timestamps:
        pos = np.where(df.index == ts)[0].tolist()
        duplicate_locations.append((str(ts), pos))
        pos_str = ", ".join(str(p) for p in pos[:20])
        more = "" if len(pos) <= 20 else f" ... (+{len(pos) - 20} más)"
        print(f"  - {ts}: posiciones [{pos_str}]{more}")

    if duplicate_timestamps > max_show:
        print(f"  ... (+{duplicate_timestamps - max_show} timestamps duplicados más)")

    return {
        "duplicate_rows": dup_rows,
        "duplicate_timestamps": duplicate_timestamps,
        "duplicate_locations": duplicate_locations
    }


def clean_dataframe(df, label, station_name=None, transect_clean=None):
    """
    Limpia un DataFrame:
      1. Convierte columnas numéricas a numérico
      2. Rellena metadatos de estación y transecto si faltan
      3. Muestra NaNs y duplicados
      4. Elimina índices NaT
      5. Elimina duplicados de datetime
      6. Elimina todas las filas con cualquier NaN
    """
    df = df.copy()

    # Relleno de metadatos para no perder trazabilidad
    if station_name is not None:
        if "Estacion" not in df.columns:
            df["Estacion"] = station_name
        else:
            df["Estacion"] = df["Estacion"].fillna(station_name)

    if transect_clean is not None and "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].fillna(transect_clean.replace("_", " "))

    # Convertir columnas de interés a numérico
    df = convert_numeric_columns(df)

    # Reporte antes de limpiar
    original_shape = df.shape
    total_nans_before = int(df.isna().sum().sum())
    duplicate_info = report_duplicates(df, label)
    nans_before = report_nans(df, label)

    # Eliminar índices NaT
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Filas con índice datetime inválido (NaT): {nat_count}")
        nat_positions = np.where(nat_mask)[0].tolist()
        print(f"[{label}] Posiciones de NaT: {nat_positions[:20]}{' ...' if len(nat_positions) > 20 else ''}")
        df = df.loc[~nat_mask].copy()

    # Ordenar por datetime antes de eliminar duplicados
    df = df.sort_index(kind="mergesort")

    # Eliminar duplicados de datetime manteniendo la primera ocurrencia
    dup_mask_after_nat = df.index.duplicated(keep="first")
    duplicated_rows_removed = int(dup_mask_after_nat.sum())
    if duplicated_rows_removed > 0:
        print(f"\n[{label}] Filas duplicadas que se eliminarán (keep='first'): {duplicated_rows_removed}")
        dup_positions = np.where(dup_mask_after_nat)[0].tolist()
        print(f"[{label}] Posiciones a eliminar por duplicado: {dup_positions[:20]}{' ...' if len(dup_positions) > 20 else ''}")
    df = df.loc[~dup_mask_after_nat].copy()

    rows_after_duplicates = df.shape[0]

    # Eliminar filas con cualquier NaN en cualquier columna
    nan_rows_mask = df.isna().any(axis=1)
    nan_rows_removed = int(nan_rows_mask.sum())
    if nan_rows_removed > 0:
        print(f"\n[{label}] Filas que contienen al menos un NaN y se eliminarán: {nan_rows_removed}")
        nan_row_positions = np.where(nan_rows_mask)[0].tolist()
        print(f"[{label}] Posiciones a eliminar por NaN: {nan_row_positions[:20]}{' ...' if len(nan_row_positions) > 20 else ''}")
    df = df.loc[~nan_rows_mask].copy()

    # Recuento final
    final_shape = df.shape

    print(f"\n[{label}] RESUMEN DE LIMPIEZA")
    print(f"  - Tamaño original: {original_shape}")
    print(f"  - NaNs detectados antes de limpiar: {nans_before}")
    print(f"  - NaNs detectados por recuento total: {total_nans_before}")
    print(f"  - Duplicados detectados antes de limpiar: {duplicate_info['duplicate_rows']} filas implicadas")
    print(f"  - Filas eliminadas por NaT: {nat_count}")
    print(f"  - Filas eliminadas por duplicados: {duplicated_rows_removed}")
    print(f"  - Filas eliminadas por NaN: {nan_rows_removed}")
    print(f"  - Tamaño tras quitar duplicados: ({rows_after_duplicates}, {original_shape[1]})")
    print(f"  - Tamaño final: {final_shape}")

    return df


def prepare_station_dataframe(df, station_name, transect_clean=None):
    """
    Prepara y limpia un DataFrame de estación.
    Mantiene columnas originales y solo limpia duplicados y NaNs.
    """
    df = df.copy()

    # Garantizar columna Estacion
    if "Estacion" not in df.columns:
        df["Estacion"] = station_name
    else:
        df["Estacion"] = df["Estacion"].fillna(station_name)

    cleaned = clean_dataframe(
        df,
        label=station_name,
        station_name=station_name,
        transect_clean=transect_clean
    )
    return cleaned


def finalize_combined_dataframe(df, label):
    """
    Limpieza final tras concatenar estaciones.
    Mantiene el mismo criterio: eliminar NaNs, NaT y duplicados residuales.
    """
    df = df.copy()

    # Asegurar índice datetime válido
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Índices NaT tras concatenación: {nat_count}")
        df = df.loc[~nat_mask].copy()

    # Orden estable
    df = df.sort_index(kind="mergesort")

    # Eliminar duplicados de datetime si los hubiera
    dup_mask = df.index.duplicated(keep="first")
    dup_count = int(dup_mask.sum())
    if dup_count > 0:
        print(f"\n[{label}] Duplicados residuales tras concatenación: {dup_count}")
        df = df.loc[~dup_mask].copy()

    # Eliminar cualquier NaN residual
    nan_rows_mask = df.isna().any(axis=1)
    nan_count = int(nan_rows_mask.sum())
    if nan_count > 0:
        print(f"[{label}] Filas con NaN residuales tras concatenación: {nan_count}")
        df = df.loc[~nan_rows_mask].copy()

    return df


# ============================================================================
# PROCESAMIENTO POR TRANSECTO
# ============================================================================

def process_by_transect():
    print("\n=== Limpieza por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return {}, {}

    transect_dict = {}
    station_to_transect = {}
    cleaned_station_data = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df_raw = load_station_data(filepath)

        if "Transecto" not in df_raw.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue

        transect = detect_transect_label(df_raw)
        if transect is None:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue

        transect_clean = str(transect).replace(" ", "_")
        station_to_transect[station_name] = transect_clean

        df_clean = prepare_station_dataframe(
            df_raw,
            station_name=station_name,
            transect_clean=transect_clean
        )

        cleaned_station_data[station_name] = df_clean

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df_clean))

    for transect_clean, station_list in transect_dict.items():
        print(f"\n=== Procesando transecto: {transect_clean} ===")

        df_concat = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_concat = finalize_combined_dataframe(df_concat, f"TRANSECTO {transect_clean}")

        original_rows = sum(df.shape[0] for _, df in station_list)
        original_cols = df_concat.shape[1]

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)

        print(f"  Tamaño combinado antes de guardar: ({original_rows}, {original_cols})")
        print(f"  Tamaño final guardado: {df_concat.shape}")
        print(f"  Guardado: {out_path}")

    return cleaned_station_data, station_to_transect


# ============================================================================
# PROCESAMIENTO GLOBAL
# ============================================================================

def process_global(cleaned_station_data, station_to_transect):
    print("\n=== Limpieza global (por estación) ===")

    if not cleaned_station_data:
        print("  No hay datos válidos para limpieza global.")
        return

    for station_name, df_station in cleaned_station_data.items():
        df_station = df_station.copy()

        # Asegurar que se mantiene la trazabilidad
        if "Estacion" not in df_station.columns:
            df_station["Estacion"] = station_name
        else:
            df_station["Estacion"] = df_station["Estacion"].ffill().bfill().fillna(station_name)

        if "Transecto" in df_station.columns:
            df_station["Transecto"] = df_station["Transecto"].ffill().bfill()

        df_station = finalize_combined_dataframe(df_station, f"ESTACION {station_name}")

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station_name}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} | tamaño final: {df_station.shape}")


# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando limpieza de datos (por transecto y global)...")
    cleaned_station_data, station_to_transect = process_by_transect()
    process_global(cleaned_station_data, station_to_transect)
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")